In [3]:
import torch
from minicons import scorer
from tqdm.auto import tqdm
import json
device = 'cpu'
lm = scorer.IncrementalLMScorer("facebook/opt-125m", device=device)

In [4]:
def encode_with_prompt_completion_format_minicons(example, tokenizer):
  
    messages = example["messages"]
    for msg in messages:
        if msg['role'] == 'user':
            question = msg['content']
        if msg['role'] == 'assistant':
            answer = msg['content']

    
    answer = answer + tokenizer.eos_token
    return question, answer

In [5]:
f = open('../../full_data/siqa_train_options_answer.json')
f1 = open('../../full_data/100_high_p0_siqa_train_options_answer_p0.json','w')

In [6]:
for line in f:
    d = json.loads(line)
    question, answer = encode_with_prompt_completion_format_minicons(d, lm.tokenizer)
    prefixes = [
        question,
    ]

    # these can be your answers
    continuations = [
        answer,
    ]
    likelihood = lm.conditional_score(prefixes, continuations, reduction=lambda x: (x.mean()).exp().item())
    perplexity = lm.conditional_score(prefixes, continuations, reduction=lambda x: (-x.mean()).exp().item())
    d['perplexity'] = perplexity[0]
    d['likelihood'] = likelihood[0]
    json.dump(d, f1)
    f1.write('\n')
